In [ ]:
import kagglehub
import pandas as pd
from datasets import Dataset

# Download dataset automatically (no manual download)
path = kagglehub.dataset_download("dharshiyanacc/spam-ham-and-phishing-message-dataset-for-nlp")

# Load CSV
import os
files = os.listdir(path)
print(files)

# Find the CSV file
csv_path = [f for f in files if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(path, csv_path))

df.head()

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

def load_model(save_dir):
    model = AutoModelForSequenceClassification.from_pretrained(save_dir)
    tokenizer = AutoTokenizer.from_pretrained(save_dir)

    model = model.to("cuda" if torch.cuda.is_available() else "cpu")

    return model, tokenizer

In [ ]:
loaded_models = {}

for name, path in save_paths.items():
    print(f"Loading {name} from {path}")

    model, tokenizer = load_model(path)
    loaded_models[name] = (model, tokenizer)

In [ ]:
import OpenAttack as oa
import torch.nn.functional as F

class TransformerClassifier(oa.Classifier):
    def __init__(self, model, tokenizer):
        self.model = model.cuda()
        self.tokenizer = tokenizer

    def get_prob(self, input_):
        inputs = self.tokenizer(
            input_,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to("cuda")

        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = F.softmax(outputs.logits, dim=-1)

        return probs.cpu().numpy()

    def get_pred(self, input_):
        return self.get_prob(input_).argmax(axis=1)

In [ ]:
attackers = {
    "textfooler": oa.attackers.TextFoolerAttacker(),
    "deepwordbug": oa.attackers.DeepWordBugAttacker(),
    "uat": oa.attackers.UATAttacker()   # Universal Adversarial Triggers
}

def run_multiple_attacks(model, tokenizer, dataset, attackers, num_samples=100):
    clf = TransformerClassifier(model, tokenizer)

    results = {}

    for attack_name, attacker in attackers.items():
        print(f"\nRunning {attack_name} attack...\n")

        correct = 0
        attacked_success = 0

        for i in range(num_samples):
            text = dataset[i][TEXT_COL]
            true_label = dataset[i][LABEL_COL]

            pred = clf.get_pred([text])[0]

            # Only attack correctly classified samples
            if pred == true_label:
                correct += 1

                try:
                    result = attacker.attack(clf, text, pred)

                    if result is not None:
                        attacked_success += 1

                except Exception as e:
                    continue  # skip failures safely

        success_rate = attacked_success / correct if correct else 0

        results[attack_name] = {
            "original_correct": correct,
            "attack_success": attacked_success,
            "success_rate": success_rate
        }

        print(f"{attack_name} → Success Rate: {success_rate:.4f}")

    return results

In [ ]:
all_results = {}

for name, (model, tokenizer) in loaded_models.items():
    print(f"\nAttacking {name}...\n")

    results = run_multiple_attacks(
        model,
        tokenizer,
        test_ds,
        attackers,
        num_samples=50
    )
    all_results[name] = results

In [ ]:
import pandas as pd

rows = []

for model_name, attack_dict in all_results.items():
    for attack_name, res in attack_dict.items():
        rows.append({
            "Model": model_name,
            "Attack": attack_name,
            "Success Rate": res["success_rate"]
        })

df_results = pd.DataFrame(rows)
print(df_results)